<a href="https://colab.research.google.com/github/zahraamiricivil1997-rgb/GTA-Meteorological-Data/blob/main/Traffic_Density.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# STEP 1 — Connect Google Drive & Create Project Folders
# GTA Hourly Traffic Data | 2024–2025
# ============================================================

from google.colab import drive
import os

# Connect Google Drive
drive.mount('/content/drive')

# Main project folder
BASE_DIR = "/content/drive/MyDrive/GTA_NO2_Project/Traffic_Data"

# Separate folders for GTA regions
folders = [
    "Toronto",
    "Peel",
    "York",
    "Durham",
    "Halton",
    "MTO_Highways"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("✅ Google Drive connected")
print("✅ Traffic project folders created")
print("\nMain folder:")
print(BASE_DIR)

print("\nFolders:")
for folder in folders:
    print(" -", folder)

Mounted at /content/drive
✅ Google Drive connected
✅ Traffic project folders created

Main folder:
/content/drive/MyDrive/GTA_NO2_Project/Traffic_Data

Folders:
 - Toronto
 - Peel
 - York
 - Durham
 - Halton
 - MTO_Highways


In [2]:
# ============================================================
# STEP 2 — Check Toronto Open Data
# Traffic Volumes at Intersections for All Modes
# ============================================================

import requests
import json

# Toronto Open Data CKAN API
PACKAGE_ID = "traffic-volumes-at-intersections-for-all-modes"

url = (
    "https://ckan0.cf.opendata.inter.prod-toronto.ca/"
    "api/3/action/package_show?id=" + PACKAGE_ID
)

response = requests.get(url, timeout=60)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    data = response.json()

    if data.get("success"):
        dataset = data["result"]

        print("\n✅ Dataset found")
        print("Title:", dataset.get("title"))
        print("Last modified:", dataset.get("metadata_modified"))

        print("\nAvailable resources:")
        print("-" * 80)

        resources = dataset.get("resources", [])

        for i, r in enumerate(resources, start=1):
            print(f"\nResource {i}")
            print("Name   :", r.get("name"))
            print("Format :", r.get("format"))
            print("URL    :", r.get("url"))
            print("Size   :", r.get("size"))

    else:
        print("❌ API responded but dataset was not returned.")

else:
    print("❌ Could not connect to Toronto Open Data.")
    print(response.text[:500])

HTTP status: 200

✅ Dataset found
Title: Traffic Volumes - Multimodal Intersection Turning Movement Counts
Last modified: 2026-09-30T22:47:59.778877

Available resources:
--------------------------------------------------------------------------------

Resource 1
Name   : tmc_most_recent_summary_data
Format : CSV
URL    : https://ckan0.cf.opendata.inter.prod-toronto.ca/datastore/dump/6afa3b1f-f6a5-4235-8bd6-7568411c19f4
Size   : None

Resource 2
Name   : tmc_summary_data
Format : CSV
URL    : https://ckan0.cf.opendata.inter.prod-toronto.ca/datastore/dump/1364bffa-29a3-4c39-af8a-925d8ca7bf1f
Size   : None

Resource 3
Name   : tmc_raw_data_1980_1989
Format : CSV
URL    : https://ckan0.cf.opendata.inter.prod-toronto.ca/datastore/dump/4402bb65-3936-49db-bf51-ec973381a5a3
Size   : None

Resource 4
Name   : tmc_raw_data_1990_1999
Format : CSV
URL    : https://ckan0.cf.opendata.inter.prod-toronto.ca/datastore/dump/ce5ea7df-ee82-450e-aceb-2977ceeef66a
Size   : None

Resource 5
Name   : tmc_raw

In [3]:
# ============================================================
# STEP 3 — Find Toronto Traffic Raw Data for 2020–2029
# ============================================================

resources = dataset.get("resources", [])

target_resource = None

print("Searching for 2020–2029 raw traffic data...\n")

for r in resources:

    name = str(r.get("name", "")).lower()

    if "raw" in name and ("2020" in name or "2029" in name):

        target_resource = r

        print("✅ TARGET RESOURCE FOUND")
        print("-" * 60)

        print("Name   :", r.get("name"))
        print("Format :", r.get("format"))
        print("URL    :", r.get("url"))
        print("Size   :", r.get("size"))

        break


if target_resource is None:
    print("❌ 2020–2029 raw traffic resource was not found.")

Searching for 2020–2029 raw traffic data...

✅ TARGET RESOURCE FOUND
------------------------------------------------------------
Name   : tmc_raw_data_2020_2029
Format : CSV
URL    : https://ckan0.cf.opendata.inter.prod-toronto.ca/datastore/dump/262469c2-abfe-4756-9068-4ea5c7ba1af7
Size   : None


In [4]:
# ============================================================
# STEP 4 — Read Toronto Traffic Data (2020–2029)
# Inspect columns and sample records
# ============================================================

import pandas as pd
import requests
from io import BytesIO

# Direct URL found in STEP 3
traffic_url = target_resource["url"]

print("Reading Toronto traffic data...")
print("Source:", target_resource["name"])
print("-" * 60)

# Download CSV into memory
response = requests.get(traffic_url, timeout=300)
response.raise_for_status()

print(f"✅ Download completed: {len(response.content)/1024/1024:.2f} MB")

# Read CSV
df_traffic = pd.read_csv(
    BytesIO(response.content),
    low_memory=False
)

print("\n✅ CSV successfully loaded")

print("\nDataset shape:")
print("Rows   :", len(df_traffic))
print("Columns:", len(df_traffic.columns))

print("\nCOLUMN NAMES")
print("=" * 60)

for i, col in enumerate(df_traffic.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nFIRST 5 ROWS")
print("=" * 60)

display(df_traffic.head())

Reading Toronto traffic data...
Source: tmc_raw_data_2020_2029
------------------------------------------------------------
✅ Download completed: 83.41 MB

✅ CSV successfully loaded

Dataset shape:
Rows   : 361443
Columns: 55

COLUMN NAMES
01. _id
02. count_id
03. count_date
04. location_name
05. longitude
06. latitude
07. centreline_type
08. centreline_id
09. px
10. start_time
11. end_time
12. n_appr_cars_r
13. n_appr_cars_t
14. n_appr_cars_l
15. s_appr_cars_r
16. s_appr_cars_t
17. s_appr_cars_l
18. e_appr_cars_r
19. e_appr_cars_t
20. e_appr_cars_l
21. w_appr_cars_r
22. w_appr_cars_t
23. w_appr_cars_l
24. n_appr_truck_r
25. n_appr_truck_t
26. n_appr_truck_l
27. s_appr_truck_r
28. s_appr_truck_t
29. s_appr_truck_l
30. e_appr_truck_r
31. e_appr_truck_t
32. e_appr_truck_l
33. w_appr_truck_r
34. w_appr_truck_t
35. w_appr_truck_l
36. n_appr_bus_r
37. n_appr_bus_t
38. n_appr_bus_l
39. s_appr_bus_r
40. s_appr_bus_t
41. s_appr_bus_l
42. e_appr_bus_r
43. e_appr_bus_t
44. e_appr_bus_l
45. w_app

,_id,count_id,count_date,location_name,longitude,latitude,centreline_type,centreline_id,px,start_time,...,w_appr_bus_t,w_appr_bus_l,n_appr_peds,s_appr_peds,e_appr_peds,w_appr_peds,n_appr_bike,s_appr_bike,e_appr_bike,w_appr_bike
0,1,105756,2023-09-26,Bloor St W / Palmerston Blvd / Palmerston Ave,-79.413647,43.664611,2,13464470,1238.0,2023-09-26T06:30:00,...,4,0,8,3,1,0,5,1,5,19
1,2,105756,2023-09-26,Bloor St W / Palmerston Blvd / Palmerston Ave,-79.413647,43.664611,2,13464470,1238.0,2023-09-26T07:00:00,...,1,0,8,9,5,3,1,1,7,24
2,3,105756,2023-09-26,Bloor St W / Palmerston Blvd / Palmerston Ave,-79.413647,43.664611,2,13464470,1238.0,2023-09-26T08:45:00,...,1,0,27,38,17,31,18,9,23,103
3,4,105756,2023-09-26,Bloor St W / Palmerston Blvd / Palmerston Ave,-79.413647,43.664611,2,13464470,1238.0,2023-09-26T11:30:00,...,6,0,76,52,20,21,4,6,15,38
4,5,105756,2023-09-26,Bloor St W / Palmerston Blvd / Palmerston Ave,-79.413647,43.664611,2,13464470,1238.0,2023-09-26T12:00:00,...,1,0,73,74,17,24,3,7,22,35


In [5]:
# ============================================================
# STEP 5 — Check Availability of 2024 and 2025 Traffic Data
# ============================================================

import pandas as pd

# Convert count_date to datetime
df_traffic["count_date"] = pd.to_datetime(
    df_traffic["count_date"],
    errors="coerce"
)

# Create year column
df_traffic["year"] = df_traffic["count_date"].dt.year

print("=" * 60)
print("AVAILABLE YEARS")
print("=" * 60)

year_counts = (
    df_traffic["year"]
    .value_counts()
    .sort_index()
)

print(year_counts)

# ------------------------------------------------------------
# Check target years
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TARGET YEARS: 2024 & 2025")
print("=" * 60)

for year in [2024, 2025]:

    temp = df_traffic[df_traffic["year"] == year]

    print(f"\nYEAR {year}")
    print("-" * 30)

    print("Rows:", len(temp))

    if len(temp) > 0:

        print("First date:", temp["count_date"].min().date())
        print("Last date :", temp["count_date"].max().date())

        print(
            "Unique locations:",
            temp["location_name"].nunique()
        )

print("\n✅ YEAR AVAILABILITY CHECK FINISHED")

AVAILABLE YEARS
year
2020    28000
2021    36448
2022    69920
2023    36832
2024    79084
2025    75263
2026    35896
Name: count, dtype: int64

TARGET YEARS: 2024 & 2025

YEAR 2024
------------------------------
Rows: 79084
First date: 2024-01-16
Last date : 2024-12-18
Unique locations: 1091

YEAR 2025
------------------------------
Rows: 75263
First date: 2025-01-07
Last date : 2025-12-11
Unique locations: 1065

✅ YEAR AVAILABILITY CHECK FINISHED


In [6]:
# ============================================================
# STEP 6 — Inspect Time Resolution of Toronto Traffic Counts
# Target: 2024–2025
# ============================================================

import pandas as pd

# Keep only target years
df_target = df_traffic[
    df_traffic["year"].isin([2024, 2025])
].copy()

# Convert start_time to datetime
df_target["start_time"] = pd.to_datetime(
    df_target["start_time"],
    errors="coerce"
)

print("=" * 60)
print("TIME STRUCTURE — 2024 & 2025")
print("=" * 60)

print("\nValid timestamps:")
print(df_target["start_time"].notna().sum())

# ------------------------------------------------------------
# Show most common minute values
# ------------------------------------------------------------

print("\nMinute values used in the dataset:")
print(
    df_target["start_time"]
    .dt.minute
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# Hours represented
# ------------------------------------------------------------

print("\nHours represented in the dataset:")
print(
    df_target["start_time"]
    .dt.hour
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# Sample timestamps
# ------------------------------------------------------------

print("\nSample timestamps:")
print(
    df_target[
        ["count_date", "start_time", "location_name"]
    ]
    .head(20)
    .to_string(index=False)
)

print("\n✅ TIME STRUCTURE CHECK FINISHED")

TIME STRUCTURE — 2024 & 2025

Valid timestamps:
154347

Minute values used in the dataset:
start_time
0     38587
15    38588
30    38587
45    38585
Name: count, dtype: int64

Hours represented in the dataset:
start_time
6     10923
7     11028
8     11048
9     11028
10    11048
11    11040
12    11016
13    11044
14    11046
15    11014
16    11048
17    11048
18    11008
19    11008
Name: count, dtype: int64

Sample timestamps:
count_date          start_time                 location_name
2024-01-17 2024-01-17 06:00:00         Pelham Ave / Osler St
2024-01-17 2024-01-17 10:30:00         Pelham Ave / Osler St
2024-01-17 2024-01-17 13:45:00         Pelham Ave / Osler St
2024-01-17 2024-01-17 19:30:00         Pelham Ave / Osler St
2024-11-19 2024-11-19 06:00:00       Queen St E / Empire Ave
2024-11-19 2024-11-19 08:15:00       Queen St E / Empire Ave
2024-11-19 2024-11-19 13:30:00       Queen St E / Empire Ave
2024-11-19 2024-11-19 15:00:00       Queen St E / Empire Ave
2024-01-31 2024

In [7]:
# ============================================================
# STEP 7 — Inspect Traffic Count Columns
# Toronto | 2024–2025
# Purpose: Identify vehicle, bike and pedestrian variables
# ============================================================

print("=" * 70)
print("TRAFFIC COUNT COLUMNS")
print("=" * 70)

# Show all columns with their data type
for i, col in enumerate(df_target.columns, start=1):

    dtype = df_target[col].dtype

    print(f"{i:02d}. {col:<25} | {dtype}")


# ============================================================
# Find likely movement/count columns
# ============================================================

keywords = [
    "car",
    "truck",
    "bus",
    "bike",
    "ped",
    "appr",
    "light",
    "heavy"
]

traffic_columns = [
    col for col in df_target.columns
    if any(word in col.lower() for word in keywords)
]

print("\n" + "=" * 70)
print("POTENTIAL TRAFFIC VARIABLES")
print("=" * 70)

for col in traffic_columns:
    print(col)


# ============================================================
# Show sample values
# ============================================================

print("\n" + "=" * 70)
print("SAMPLE VALUES")
print("=" * 70)

display(
    df_target[traffic_columns].head(10)
)

print("\n✅ TRAFFIC COLUMN CHECK FINISHED")

TRAFFIC COUNT COLUMNS
01. _id                       | int64
02. count_id                  | int64
03. count_date                | datetime64[ns]
04. location_name             | object
05. longitude                 | float64
06. latitude                  | float64
07. centreline_type           | int64
08. centreline_id             | int64
09. px                        | float64
10. start_time                | datetime64[ns]
11. end_time                  | object
12. n_appr_cars_r             | int64
13. n_appr_cars_t             | int64
14. n_appr_cars_l             | int64
15. s_appr_cars_r             | int64
16. s_appr_cars_t             | int64
17. s_appr_cars_l             | int64
18. e_appr_cars_r             | int64
19. e_appr_cars_t             | int64
20. e_appr_cars_l             | int64
21. w_appr_cars_r             | int64
22. w_appr_cars_t             | int64
23. w_appr_cars_l             | int64
24. n_appr_truck_r            | int64
25. n_appr_truck_t            | int64
26

,n_appr_cars_r,n_appr_cars_t,n_appr_cars_l,s_appr_cars_r,s_appr_cars_t,s_appr_cars_l,e_appr_cars_r,e_appr_cars_t,e_appr_cars_l,w_appr_cars_r,...,w_appr_bus_t,w_appr_bus_l,n_appr_peds,s_appr_peds,e_appr_peds,w_appr_peds,n_appr_bike,s_appr_bike,e_appr_bike,w_appr_bike
118,0,4,0,0,0,0,3,0,0,0,...,0,0,0,0,0,2,0,0,0,0
119,0,11,3,6,8,2,1,0,6,0,...,0,0,1,1,4,11,1,0,0,0
120,0,6,5,4,13,0,9,0,4,0,...,0,0,1,3,2,7,0,3,1,0
121,0,10,0,5,10,0,3,0,0,0,...,0,0,0,0,1,1,0,2,2,0
132,0,0,0,1,0,2,0,15,0,0,...,0,0,5,0,0,0,0,0,0,2
133,0,0,0,5,0,4,0,178,0,0,...,1,0,30,21,11,2,0,0,8,4
134,0,0,0,4,0,3,0,34,0,0,...,0,0,43,28,3,2,0,2,8,5
135,0,0,0,23,0,1,0,47,0,0,...,1,0,39,25,14,1,0,0,8,7
749,0,63,5,6,39,1,9,2,12,2,...,0,0,2,28,42,20,27,8,5,2
750,2,63,12,4,29,3,14,1,10,2,...,0,0,3,30,51,24,24,6,5,0



✅ TRAFFIC COLUMN CHECK FINISHED


In [8]:
# ============================================================
# STEP 8 — Identify Motor-Vehicle Count Columns
# Toronto | 2024–2025
# ============================================================

# ------------------------------------------------------------
# 1. Find all directional approach columns
# ------------------------------------------------------------

approach_cols = [
    col for col in df_target.columns
    if "_appr_" in col.lower()
]

print("=" * 70)
print("ALL APPROACH COUNT COLUMNS")
print("=" * 70)

for col in approach_cols:
    print(col)


# ------------------------------------------------------------
# 2. Separate non-motorized columns
# ------------------------------------------------------------

bike_cols = [
    col for col in approach_cols
    if "bike" in col.lower()
]

ped_cols = [
    col for col in approach_cols
    if "ped" in col.lower()
]


# ------------------------------------------------------------
# 3. Motor vehicle columns
# Exclude bikes and pedestrians
# ------------------------------------------------------------

vehicle_cols = [
    col for col in approach_cols
    if "bike" not in col.lower()
    and "ped" not in col.lower()
]


# ------------------------------------------------------------
# 4. Display classification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MOTOR VEHICLE COLUMNS")
print("=" * 70)

for col in vehicle_cols:
    print(col)

print("\nNumber of motor-vehicle columns:", len(vehicle_cols))


print("\n" + "=" * 70)
print("BICYCLE COLUMNS")
print("=" * 70)

for col in bike_cols:
    print(col)


print("\n" + "=" * 70)
print("PEDESTRIAN COLUMNS")
print("=" * 70)

for col in ped_cols:
    print(col)


print("\n✅ VEHICLE COLUMN IDENTIFICATION FINISHED")

ALL APPROACH COUNT COLUMNS
n_appr_cars_r
n_appr_cars_t
n_appr_cars_l
s_appr_cars_r
s_appr_cars_t
s_appr_cars_l
e_appr_cars_r
e_appr_cars_t
e_appr_cars_l
w_appr_cars_r
w_appr_cars_t
w_appr_cars_l
n_appr_truck_r
n_appr_truck_t
n_appr_truck_l
s_appr_truck_r
s_appr_truck_t
s_appr_truck_l
e_appr_truck_r
e_appr_truck_t
e_appr_truck_l
w_appr_truck_r
w_appr_truck_t
w_appr_truck_l
n_appr_bus_r
n_appr_bus_t
n_appr_bus_l
s_appr_bus_r
s_appr_bus_t
s_appr_bus_l
e_appr_bus_r
e_appr_bus_t
e_appr_bus_l
w_appr_bus_r
w_appr_bus_t
w_appr_bus_l
n_appr_peds
s_appr_peds
e_appr_peds
w_appr_peds
n_appr_bike
s_appr_bike
e_appr_bike
w_appr_bike

MOTOR VEHICLE COLUMNS
n_appr_cars_r
n_appr_cars_t
n_appr_cars_l
s_appr_cars_r
s_appr_cars_t
s_appr_cars_l
e_appr_cars_r
e_appr_cars_t
e_appr_cars_l
w_appr_cars_r
w_appr_cars_t
w_appr_cars_l
n_appr_truck_r
n_appr_truck_t
n_appr_truck_l
s_appr_truck_r
s_appr_truck_t
s_appr_truck_l
e_appr_truck_r
e_appr_truck_t
e_appr_truck_l
w_appr_truck_r
w_appr_truck_t
w_appr_truck_l
n_

In [9]:
# ============================================================
# STEP 9 — Calculate Motor Vehicle Volume per Record
# Toronto | 2024–2025
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Make sure vehicle columns are numeric
# ------------------------------------------------------------

for col in vehicle_cols:
    df_target[col] = pd.to_numeric(
        df_target[col],
        errors="coerce"
    ).fillna(0)


# ------------------------------------------------------------
# 2. Calculate total motor vehicles for each record
# ------------------------------------------------------------

df_target["vehicle_volume"] = df_target[vehicle_cols].sum(axis=1)


# ------------------------------------------------------------
# 3. Keep useful information for inspection
# ------------------------------------------------------------

check_cols = [
    "count_date",
    "start_time",
    "location_name",
    "latitude",
    "longitude",
    "vehicle_volume"
]

print("=" * 70)
print("MOTOR VEHICLE VOLUME CREATED")
print("=" * 70)

display(
    df_target[check_cols].head(20)
)


# ------------------------------------------------------------
# 4. Basic statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VEHICLE VOLUME STATISTICS")
print("=" * 70)

print(df_target["vehicle_volume"].describe())


# ------------------------------------------------------------
# 5. Check duplicate location + timestamp records
# ------------------------------------------------------------

duplicates = df_target.duplicated(
    subset=["location_name", "start_time"],
    keep=False
)

print("\nDuplicate location/timestamp rows:",
      duplicates.sum())


# ------------------------------------------------------------
# 6. Show example from one location/day
# ------------------------------------------------------------

sample_location = df_target["location_name"].dropna().iloc[0]
sample_date = df_target.loc[
    df_target["location_name"] == sample_location,
    "count_date"
].iloc[0]

sample = df_target[
    (df_target["location_name"] == sample_location) &
    (df_target["count_date"] == sample_date)
][[
    "start_time",
    "vehicle_volume"
]].sort_values("start_time")

print("\nExample location:")
print(sample_location)

print("\nExample date:")
print(sample_date)

display(sample)

print("\n✅ MOTOR VEHICLE VOLUME CHECK FINISHED")

MOTOR VEHICLE VOLUME CREATED


,count_date,start_time,location_name,latitude,longitude,vehicle_volume
118,2024-01-17,2024-01-17 06:00:00,Pelham Ave / Osler St,43.667583,-79.458313,7
119,2024-01-17,2024-01-17 10:30:00,Pelham Ave / Osler St,43.667583,-79.458313,39
120,2024-01-17,2024-01-17 13:45:00,Pelham Ave / Osler St,43.667583,-79.458313,42
121,2024-01-17,2024-01-17 19:30:00,Pelham Ave / Osler St,43.667583,-79.458313,28
132,2024-11-19,2024-11-19 06:00:00,Queen St E / Empire Ave,43.660086,-79.344459,30
133,2024-11-19,2024-11-19 08:15:00,Queen St E / Empire Ave,43.660086,-79.344459,240
134,2024-11-19,2024-11-19 13:30:00,Queen St E / Empire Ave,43.660086,-79.344459,92
135,2024-11-19,2024-11-19 15:00:00,Queen St E / Empire Ave,43.660086,-79.344459,147
749,2024-01-31,2024-01-31 09:00:00,Sherbourne St / Earl St,43.668711,-79.375354,149
750,2024-01-31,2024-01-31 09:15:00,Sherbourne St / Earl St,43.668711,-79.375354,149



VEHICLE VOLUME STATISTICS
count    154347.000000
mean        370.971085
std         236.560118
min           0.000000
25%         201.000000
50%         341.000000
75%         500.000000
max        2404.000000
Name: vehicle_volume, dtype: float64

Duplicate location/timestamp rows: 672

Example location:
Pelham Ave / Osler St

Example date:
2024-01-17 00:00:00


,start_time,vehicle_volume
118,2024-01-17 06:00:00,7
179711,2024-01-17 06:15:00,13
266447,2024-01-17 06:30:00,13
12610,2024-01-17 06:45:00,20
348253,2024-01-17 07:00:00,19
242276,2024-01-17 07:15:00,25
242277,2024-01-17 07:30:00,22
218983,2024-01-17 07:45:00,45
320664,2024-01-17 08:00:00,63
256615,2024-01-17 08:15:00,83



✅ MOTOR VEHICLE VOLUME CHECK FINISHED


In [10]:
# ============================================================
# STEP 10 — Aggregate Traffic Counts to Hourly Resolution
# Toronto | 2024–2025
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Make sure start_time is datetime
# ------------------------------------------------------------

df_target["start_time"] = pd.to_datetime(
    df_target["start_time"],
    errors="coerce"
)

# Remove rows without valid timestamp
df_hourly_base = df_target.dropna(
    subset=["start_time", "latitude", "longitude"]
).copy()


# ------------------------------------------------------------
# 2. Create hourly timestamp
# Example:
# 08:00, 08:15, 08:30, 08:45 -> 08:00
# ------------------------------------------------------------

df_hourly_base["datetime_hour"] = (
    df_hourly_base["start_time"].dt.floor("h")
)


# ------------------------------------------------------------
# 3. Aggregate traffic counts by location and hour
# ------------------------------------------------------------

hourly_traffic = (
    df_hourly_base
    .groupby(
        [
            "location_name",
            "latitude",
            "longitude",
            "datetime_hour"
        ],
        as_index=False
    )
    .agg(
        traffic_volume=("vehicle_volume", "sum"),
        records_in_hour=("vehicle_volume", "size")
    )
)


# ------------------------------------------------------------
# 4. Add useful temporal variables
# ------------------------------------------------------------

hourly_traffic["date"] = hourly_traffic["datetime_hour"].dt.date
hourly_traffic["year"] = hourly_traffic["datetime_hour"].dt.year
hourly_traffic["month"] = hourly_traffic["datetime_hour"].dt.month
hourly_traffic["day"] = hourly_traffic["datetime_hour"].dt.day
hourly_traffic["hour"] = hourly_traffic["datetime_hour"].dt.hour


# ------------------------------------------------------------
# 5. Reorder columns
# ------------------------------------------------------------

hourly_traffic = hourly_traffic[
    [
        "datetime_hour",
        "date",
        "year",
        "month",
        "day",
        "hour",
        "location_name",
        "latitude",
        "longitude",
        "traffic_volume",
        "records_in_hour"
    ]
]


# ------------------------------------------------------------
# 6. Sort
# ------------------------------------------------------------

hourly_traffic = hourly_traffic.sort_values(
    ["datetime_hour", "location_name"]
).reset_index(drop=True)


# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

print("=" * 70)
print("TORONTO HOURLY TRAFFIC DATA CREATED")
print("=" * 70)

print("\nTotal hourly records:")
print(len(hourly_traffic))

print("\n2024 hourly records:")
print((hourly_traffic["year"] == 2024).sum())

print("\n2025 hourly records:")
print((hourly_traffic["year"] == 2025).sum())

print("\nRecords per hour distribution:")
print(
    hourly_traffic["records_in_hour"]
    .value_counts()
    .sort_index()
)

print("\nSample:")
display(hourly_traffic.head(20))

print("\n✅ HOURLY AGGREGATION FINISHED")

TORONTO HOURLY TRAFFIC DATA CREATED

Total hourly records:
38515

2024 hourly records:
19741

2025 hourly records:
18774

Records per hour distribution:
records_in_hour
2       22
3        5
4    38404
8       84
Name: count, dtype: int64

Sample:


,datetime_hour,date,year,month,day,hour,location_name,latitude,longitude,traffic_volume,records_in_hour
0,2024-01-16 06:00:00,2024-01-16,2024,1,16,6,King St W / Strachan Ave,43.642043,-79.411971,574,4
1,2024-01-16 06:00:00,2024-01-16,2024,1,16,6,Saskatchewan Rd / Princes' Blvd / Manitoba Dr,43.632635,-79.422298,74,4
2,2024-01-16 06:00:00,2024-01-16,2024,1,16,6,Springhurst Ave / Dufferin St,43.634862,-79.425794,484,4
3,2024-01-16 07:00:00,2024-01-16,2024,1,16,7,King St W / Strachan Ave,43.642043,-79.411971,843,4
4,2024-01-16 07:00:00,2024-01-16,2024,1,16,7,Saskatchewan Rd / Princes' Blvd / Manitoba Dr,43.632635,-79.422298,106,4
5,2024-01-16 07:00:00,2024-01-16,2024,1,16,7,Springhurst Ave / Dufferin St,43.634862,-79.425794,741,4
6,2024-01-16 08:00:00,2024-01-16,2024,1,16,8,King St W / Strachan Ave,43.642043,-79.411971,1306,4
7,2024-01-16 08:00:00,2024-01-16,2024,1,16,8,Saskatchewan Rd / Princes' Blvd / Manitoba Dr,43.632635,-79.422298,454,4
8,2024-01-16 08:00:00,2024-01-16,2024,1,16,8,Springhurst Ave / Dufferin St,43.634862,-79.425794,987,4
9,2024-01-16 09:00:00,2024-01-16,2024,1,16,9,King St W / Strachan Ave,43.642043,-79.411971,1317,4



✅ HOURLY AGGREGATION FINISHED


In [13]:
# ============================================================
# STEP 11 — Quality Control of Hourly Traffic Data
# Toronto | 2024–2025
# ============================================================

print("=" * 70)
print("HOURLY TRAFFIC QUALITY CHECK")
print("=" * 70)

# Distribution
distribution = (
    hourly_traffic["records_in_hour"]
    .value_counts()
    .sort_index()
)

print("\nRecords per hour:")
print(distribution)

# Complete hours = four 15-minute records
complete = hourly_traffic[
    hourly_traffic["records_in_hour"] == 4
].copy()

incomplete = hourly_traffic[
    hourly_traffic["records_in_hour"] != 4
].copy()

total = len(hourly_traffic)

print("\n" + "=" * 70)

print("Total hourly records:", total)
print("Complete hourly records:", len(complete))
print("Incomplete hourly records:", len(incomplete))

print(
    "Complete hours (%):",
    round(len(complete) / total * 100, 2)
)

# Check separately by year
print("\n" + "=" * 70)
print("COMPLETE HOURS BY YEAR")
print("=" * 70)

for year in [2024, 2025]:

    temp = complete[complete["year"] == year]

    print(f"\n{year}")
    print("Complete hourly records:", len(temp))
    print("Locations:", temp["location_name"].nunique())
    print("First date:", temp["datetime_hour"].min())
    print("Last date :", temp["datetime_hour"].max())

print("\n✅ QUALITY CHECK FINISHED")

HOURLY TRAFFIC QUALITY CHECK

Records per hour:
records_in_hour
2       22
3        5
4    38404
8       84
Name: count, dtype: int64

Total hourly records: 38515
Complete hourly records: 38404
Incomplete hourly records: 111
Complete hours (%): 99.71

COMPLETE HOURS BY YEAR

2024
Complete hourly records: 19673
Locations: 1088
First date: 2024-01-16 06:00:00
Last date : 2024-12-18 19:00:00

2025
Complete hourly records: 18731
Locations: 1063
First date: 2025-01-07 06:00:00
Last date : 2025-12-11 19:00:00

✅ QUALITY CHECK FINISHED


In [14]:
# ============================================================
# STEP 12 — Export Toronto Hourly Traffic Data
# Separate files for 2024 and 2025
# ============================================================

import os
import pandas as pd
from google.colab import files

# Toronto folder already created in STEP 1
TORONTO_DIR = os.path.join(BASE_DIR, "Toronto")

os.makedirs(TORONTO_DIR, exist_ok=True)

# ------------------------------------------------------------
# Separate complete hourly records by year
# ------------------------------------------------------------

traffic_2024 = complete[
    complete["year"] == 2024
].copy()

traffic_2025 = complete[
    complete["year"] == 2025
].copy()

# Sort
traffic_2024 = traffic_2024.sort_values(
    ["datetime_hour", "location_name"]
).reset_index(drop=True)

traffic_2025 = traffic_2025.sort_values(
    ["datetime_hour", "location_name"]
).reset_index(drop=True)


# ------------------------------------------------------------
# File paths
# ------------------------------------------------------------

file_2024 = os.path.join(
    TORONTO_DIR,
    "Toronto_Hourly_Traffic_2024.xlsx"
)

file_2025 = os.path.join(
    TORONTO_DIR,
    "Toronto_Hourly_Traffic_2025.xlsx"
)


# ------------------------------------------------------------
# Save Excel files
# ------------------------------------------------------------

traffic_2024.to_excel(
    file_2024,
    index=False
)

traffic_2025.to_excel(
    file_2025,
    index=False
)


# ------------------------------------------------------------
# Check files
# ------------------------------------------------------------

print("=" * 70)
print("TORONTO HOURLY TRAFFIC EXPORT")
print("=" * 70)

print("\n2024:")
print("Rows:", len(traffic_2024))
print("Locations:", traffic_2024["location_name"].nunique())
print("File:", file_2024)

print("\n2025:")
print("Rows:", len(traffic_2025))
print("Locations:", traffic_2025["location_name"].nunique())
print("File:", file_2025)

print("\n✅ Both Excel files saved safely to Google Drive")


# ------------------------------------------------------------
# Download to computer
# ------------------------------------------------------------

print("\n⬇️ Downloading 2024...")
files.download(file_2024)

print("\n⬇️ Downloading 2025...")
files.download(file_2025)


TORONTO HOURLY TRAFFIC EXPORT

2024:
Rows: 19673
Locations: 1088
File: /content/drive/MyDrive/GTA_NO2_Project/Traffic_Data/Toronto/Toronto_Hourly_Traffic_2024.xlsx

2025:
Rows: 18731
Locations: 1063
File: /content/drive/MyDrive/GTA_NO2_Project/Traffic_Data/Toronto/Toronto_Hourly_Traffic_2025.xlsx

✅ Both Excel files saved safely to Google Drive

⬇️ Downloading 2024...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


⬇️ Downloading 2025...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# ============================================================
# FINAL EXPORT — TORONTO HOURLY TRAFFIC DATA
# Years: 2024 and 2025
#
# Output includes:
# - Date
# - Hour (HH:MM:SS)
# - Location
# - Latitude / Longitude
# - Cars
# - Trucks
# - Buses
# - Bikes
# - Pedestrians
# - Total Motor Vehicles
# - Number of 15-min records used
# ============================================================

import os
import pandas as pd
from google.colab import files


# ============================================================
# 1. IDENTIFY VEHICLE-TYPE COLUMNS
# ============================================================

# All approach columns
approach_cols = [
    col for col in df_target.columns
    if "_appr_" in col.lower()
]

# ------------------------------------------------------------
# Cars
# Everything motorized that is NOT truck, bus, bike or ped
# ------------------------------------------------------------

car_cols = [
    col for col in approach_cols
    if "truck" not in col.lower()
    and "bus" not in col.lower()
    and "bike" not in col.lower()
    and "ped" not in col.lower()
]

# Trucks
truck_cols = [
    col for col in approach_cols
    if "truck" in col.lower()
]

# Buses
bus_cols = [
    col for col in approach_cols
    if "bus" in col.lower()
]

# Bikes
bike_cols = [
    col for col in approach_cols
    if "bike" in col.lower()
]

# Pedestrians
ped_cols = [
    col for col in approach_cols
    if "ped" in col.lower()
]


print("=" * 70)
print("DETECTED TRAFFIC VARIABLES")
print("=" * 70)

print("Car columns   :", len(car_cols))
print("Truck columns :", len(truck_cols))
print("Bus columns   :", len(bus_cols))
print("Bike columns  :", len(bike_cols))
print("Ped columns   :", len(ped_cols))


# ============================================================
# 2. CONVERT COUNT COLUMNS TO NUMERIC
# ============================================================

all_count_cols = (
    car_cols +
    truck_cols +
    bus_cols +
    bike_cols +
    ped_cols
)

for col in all_count_cols:
    df_target[col] = pd.to_numeric(
        df_target[col],
        errors="coerce"
    ).fillna(0)


# ============================================================
# 3. CALCULATE VEHICLE TYPES FOR EACH RAW RECORD
# ============================================================

df_target["cars"] = df_target[car_cols].sum(axis=1)

df_target["trucks"] = df_target[truck_cols].sum(axis=1)

df_target["buses"] = df_target[bus_cols].sum(axis=1)

df_target["bikes"] = df_target[bike_cols].sum(axis=1)

df_target["pedestrians"] = df_target[ped_cols].sum(axis=1)


# Total motorized traffic
df_target["total_motor_vehicles"] = (
    df_target["cars"] +
    df_target["trucks"] +
    df_target["buses"]
)


# ============================================================
# 4. PREPARE TIME
# ============================================================

df_target["start_time"] = pd.to_datetime(
    df_target["start_time"],
    errors="coerce"
)

df_work = df_target.dropna(
    subset=[
        "start_time",
        "latitude",
        "longitude"
    ]
).copy()


# Floor timestamps to beginning of hour
df_work["datetime_hour"] = (
    df_work["start_time"].dt.floor("h")
)


# ============================================================
# 5. AGGREGATE 15-MINUTE COUNTS TO HOURLY COUNTS
# ============================================================

hourly_final = (
    df_work
    .groupby(
        [
            "location_name",
            "latitude",
            "longitude",
            "datetime_hour"
        ],
        as_index=False
    )
    .agg(

        cars=("cars", "sum"),

        trucks=("trucks", "sum"),

        buses=("buses", "sum"),

        bikes=("bikes", "sum"),

        pedestrians=("pedestrians", "sum"),

        total_motor_vehicles=(
            "total_motor_vehicles",
            "sum"
        ),

        records_in_hour=(
            "total_motor_vehicles",
            "size"
        )
    )
)


# ============================================================
# 6. KEEP ONLY COMPLETE HOURS
#
# 4 records =
# 00, 15, 30, 45 minutes
# ============================================================

hourly_final = hourly_final[
    hourly_final["records_in_hour"] == 4
].copy()


# ============================================================
# 7. CREATE DATE / TIME VARIABLES
# ============================================================

hourly_final["date"] = (
    hourly_final["datetime_hour"].dt.date
)

hourly_final["year"] = (
    hourly_final["datetime_hour"].dt.year
)

hourly_final["month"] = (
    hourly_final["datetime_hour"].dt.month
)

hourly_final["day"] = (
    hourly_final["datetime_hour"].dt.day
)

# IMPORTANT:
# Hour will appear as:
# 06:00:00
# 07:00:00
# etc.

hourly_final["hour"] = (
    hourly_final["datetime_hour"]
    .dt.strftime("%H:%M:%S")
)


# ============================================================
# 8. REORDER COLUMNS
# ============================================================

hourly_final = hourly_final[
    [
        "datetime_hour",
        "date",
        "year",
        "month",
        "day",
        "hour",

        "location_name",
        "latitude",
        "longitude",

        "cars",
        "trucks",
        "buses",

        "total_motor_vehicles",

        "bikes",
        "pedestrians",

        "records_in_hour"
    ]
]


# ============================================================
# 9. SEPARATE 2024 AND 2025
# ============================================================

traffic_2024 = hourly_final[
    hourly_final["year"] == 2024
].copy()

traffic_2025 = hourly_final[
    hourly_final["year"] == 2025
].copy()


traffic_2024 = traffic_2024.sort_values(
    ["datetime_hour", "location_name"]
).reset_index(drop=True)

traffic_2025 = traffic_2025.sort_values(
    ["datetime_hour", "location_name"]
).reset_index(drop=True)


# ============================================================
# 10. SAVE TO GOOGLE DRIVE
# ============================================================

TORONTO_DIR = (
    "/content/drive/MyDrive/"
    "GTA_NO2_Project/Traffic_Data/Toronto"
)

os.makedirs(
    TORONTO_DIR,
    exist_ok=True
)


file_2024 = os.path.join(
    TORONTO_DIR,
    "Toronto_Hourly_Traffic_COMPLETE_2024.xlsx"
)

file_2025 = os.path.join(
    TORONTO_DIR,
    "Toronto_Hourly_Traffic_COMPLETE_2025.xlsx"
)


traffic_2024.to_excel(
    file_2024,
    index=False
)

traffic_2025.to_excel(
    file_2025,
    index=False
)


# ============================================================
# 11. CHECK OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("FINAL TORONTO TRAFFIC DATA")
print("=" * 70)

print("\n2024")
print("Rows      :", len(traffic_2024))
print(
    "Locations :",
    traffic_2024["location_name"].nunique()
)

print("\n2025")
print("Rows      :", len(traffic_2025))
print(
    "Locations :",
    traffic_2025["location_name"].nunique()
)


print("\nSample:")
display(
    traffic_2024.head(10)
)


print("\n✅ Files saved successfully to Google Drive")


# ============================================================
# 12. DOWNLOAD BOTH FILES
# ============================================================

print("\n⬇️ Downloading Toronto 2024...")
files.download(file_2024)

print("\n⬇️ Downloading Toronto 2025...")
files.download(file_2025)

DETECTED TRAFFIC VARIABLES
Car columns   : 12
Truck columns : 12
Bus columns   : 12
Bike columns  : 4
Ped columns   : 4

FINAL TORONTO TRAFFIC DATA

2024
Rows      : 19673
Locations : 1088

2025
Rows      : 18731
Locations : 1063

Sample:


,datetime_hour,date,year,month,day,hour,location_name,latitude,longitude,cars,trucks,buses,total_motor_vehicles,bikes,pedestrians,records_in_hour
0,2024-01-16 06:00:00,2024-01-16,2024,1,16,06:00:00,King St W / Strachan Ave,43.642043,-79.411971,532,7,35,574,16,86,4
1,2024-01-16 06:00:00,2024-01-16,2024,1,16,06:00:00,Saskatchewan Rd / Princes' Blvd / Manitoba Dr,43.632635,-79.422298,71,3,0,74,1,6,4
2,2024-01-16 06:00:00,2024-01-16,2024,1,16,06:00:00,Springhurst Ave / Dufferin St,43.634862,-79.425794,439,17,28,484,3,11,4
3,2024-01-16 07:00:00,2024-01-16,2024,1,16,07:00:00,King St W / Strachan Ave,43.642043,-79.411971,770,24,49,843,18,268,4
4,2024-01-16 07:00:00,2024-01-16,2024,1,16,07:00:00,Saskatchewan Rd / Princes' Blvd / Manitoba Dr,43.632635,-79.422298,95,11,0,106,2,16,4
5,2024-01-16 07:00:00,2024-01-16,2024,1,16,07:00:00,Springhurst Ave / Dufferin St,43.634862,-79.425794,688,21,32,741,6,21,4
6,2024-01-16 08:00:00,2024-01-16,2024,1,16,08:00:00,King St W / Strachan Ave,43.642043,-79.411971,1229,28,49,1306,35,540,4
7,2024-01-16 08:00:00,2024-01-16,2024,1,16,08:00:00,Saskatchewan Rd / Princes' Blvd / Manitoba Dr,43.632635,-79.422298,437,13,4,454,0,18,4
8,2024-01-16 08:00:00,2024-01-16,2024,1,16,08:00:00,Springhurst Ave / Dufferin St,43.634862,-79.425794,948,13,26,987,1,33,4
9,2024-01-16 09:00:00,2024-01-16,2024,1,16,09:00:00,King St W / Strachan Ave,43.642043,-79.411971,1248,23,46,1317,25,301,4



✅ Files saved successfully to Google Drive

⬇️ Downloading Toronto 2024...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


⬇️ Downloading Toronto 2025...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>